##Version 2

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer
import torch

In [102]:
file_path = "spotify_millsongdata.csv"


df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "notshrirang/spotify-million-song-dataset",
  file_path,
)

Using Colab cache for faster access to the 'spotify-million-song-dataset' dataset.
First 5 records:   artist                   song                                        link  \
0   ABBA  Ahe's My Kind Of Girl  /a/abba/ahes+my+kind+of+girl_20598417.html   
1   ABBA       Andante, Andante       /a/abba/andante+andante_20002708.html   
2   ABBA         As Good As New        /a/abba/as+good+as+new_20003033.html   
3   ABBA                   Bang                  /a/abba/bang_20598415.html   
4   ABBA       Bang-A-Boomerang      /a/abba/bang+a+boomerang_20002668.html   

                                                text  
0  Look at her face, it's a wonderful face  \r\nA...  
1  Take it easy with me, please  \r\nTouch me gen...  
2  I'll never know why I had to go  \r\nWhy I had...  
3  Making somebody happy is a question of give an...  
4  Making somebody happy is a question of give an...  


In [103]:
df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [104]:
df.drop('link',inplace=True, axis=1)

In [105]:
!pip install transformers torch

##song summarization

##Sentence embedding
Summarization did not happen properly, i will just send the lyrics

In [107]:
try:
    with torch.serialization.safe_globals([
    'pandas.core.series.Series',  # allow this global
    ]):
      song_embeddings = torch.load("song_embeddings.pt", weights_only=False)

except FileNotFoundError:
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model = SentenceTransformer(model_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    def get_song_embedding(lyrics):
        embedding = model.encode(
            lyrics,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        return embedding
    df["embeddings"] = df["text"].apply(get_song_embedding)
    torch.save(df["embeddings"], "song_embeddings.pt")

In [109]:
song_embeddings = np.array(song_embeddings)

In [110]:
song_embeddings[0].dtype

torch.float32

In [113]:
type(song_embeddings[0])

torch.Tensor

In [114]:
song_embeddings_np = np.array([t.detach().cpu().numpy() for t in song_embeddings])

In [214]:
# Example manually curated lists
rock_artists = ["Queen", "Nirvana", "The Beatles", "Foo Fighters"]
jazz_artists = ["Miles Davis", "John Coltrane", "Herbie Hancock", "Louis Armstrong"]

# Filter your dataset using these lists
arr_rock = song_embeddings_np[df["artist"].isin(rock_artists)]
arr_jazz = song_embeddings_np[df["artist"].isin(jazz_artists)]

song_hist = np.concatenate((arr_jazz, arr_rock))
labels = np.concatenate((np.zeros(len(arr_jazz)), np.ones(len(arr_rock))))

In [215]:
x = song_hist
y = labels

In [216]:
x.shape, y.shape

((715, 768), (715,))

In [ ]:

window_size = 10 # Let's look at the last 10 songs

model = Sequential([
    layers.Input(shape=(window_size, 768)), 
    layers.LSTM(256),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
model.fit(x, y, epochs=5)